# CEQ-JEPA / DCM-1 stage-1 training -- self-contained Kaggle notebook

torch + numpy only. No pip installs, no dataset dependency: the
`ceqjepa.operator` and `ceqjepa.train` modules are inlined verbatim below
(read directly off disk by `build_notebook.py` at generation time). The
`ceqjepa` package is UNTRACKED (`git ls-files ceqjepa` is empty) -- there is
no commit that names this code's provenance. Do not cite a commit SHA as
this code's provenance; state it as an untracked working tree instead.

Runs the **stage-1** objective only (`L = L_q + 0.5*L_z`), per the DCM-1
build spec: the committor head, the constant-predictor control (C1, `q_bar`),
the collapse-floor diagnostic (CFD, `||q - q_floor||_inf`), the encoder
VAR-across-examples collapse detector, and the teleport conditioning bound
(`kappa_bound`), all logged every eval. Stage 2 (decision head / hinge loss)
is gated behind the R-1 kill and is not built here.

`build_operator` carries the teleport fix (`teleport=0.0125` by default):
every transient row sends a fraction `c` of its mass onto the causally
visible absorbing states before the boundary overwrite, giving the exact
bound `||(I-Q)^-1||_inf <= 80*(1+6e-6)` in float32 (`op.KAPPA_DESIGN_BOUND_F32`)
whenever index 0 is absorbing -- which it always is here (`torch.arange(NA)`).
That is a bound, not a promise the run stays comfortably under it: a long
soak (measured on a prior run) reached kappa 75.6954 by step 1180 and was
STILL CLIMBING, with float32 rounding able to push a further ~1e-4 past 80.
**D8 fix:** `evaluate()` therefore only WARNS when `kappa_bound` exceeds
`KAPPA_CEILING` -- it never asserts or aborts. A T4 quota should never be
spent watching a hard assert kill a healthy run mid-flight. `committor()`
and `q_floor_closed_form()` still raise `SingularTransientBlockError` on a
genuine defect (e.g. NaN logits, or an armed `kappa_max` at teleport=0);
nothing in this notebook catches that exception -- it is a bug to surface,
not a per-step event to paper over. (The kernel has no `.git` -- there is
nothing to print at runtime; the honesty fix is in this cell's own prose,
not a live git call.)

**D7 fix:** the training loop below writes an atomic checkpoint to
`/kaggle/working` every `CKPT_EVERY` steps (`save_checkpoint()`, inlined from
`train.py`: write to `<out>.tmp`, then `os.replace()` onto `<out>` -- a
9-hour Kaggle session timeout, or any kill, leaves the last periodic
checkpoint on disk under `/kaggle/working`, which Kaggle persists as kernel
output, instead of leaving nothing (the failure mode of the run this fixes:
a prior soak reached step 1840 healthy and left zero checkpoints because the
only `torch.save` ran after the loop, and the process was killed first).

In [ ]:
import time, json, platform
import torch
print("torch", torch.__version__, "cuda available:", torch.cuda.is_available())
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)


## `ceqjepa.operator` (inlined verbatim)

In [ ]:
"""
CEQ-JEPA core operator.

The declared-real math (frozen; see docs/canon/08_ARCHITECTURE.md and the
DCM-1 build spec):

    P    = causal row-stochastic attention matrix, softmax over j <= i.
           Declared absorbing rows are overwritten to identity ROWS of the
           SAME matrix, AFTER the softmax: P[a, :] = e_a for a in the
           absorbing set A. This is a rank-preserving edit, never a second
           matrix, and it carries no gradient through the overwritten rows.
    z    = (I - g*P)^{-1} Vtilde, solved by ONE lower-triangular forward
           substitution -- exact, because P is causal (row i reads only
           j <= i, so P and I - g*P are lower triangular).
    O    = (1 - g) * P @ z                                   (the state read)

    Boundary rows split P into canonical transient/absorbing blocks
    (index order preserved, so both blocks stay triangular):
        Q = P[T, T],  R = P[T, A]
    and the committor -- the probability of being absorbed into each member
    of A, starting from each transient vertex -- is the SECOND, separate
    exact solve:
        q^(bullet)_T = (I - Q)^{-1} R            (also one triangular solve)
    q attains 0 and 1 exactly; it is a probability, never a logit.

    q_floor is the committor field of the UNIFORM causal chain (softmax of
    an all-zeros logit matrix under the causal mask) -- computable in
    closed form from (n, absorbing indices) alone, no encoder, no solve.
    It is the collapse floor: a collapsed encoder produces exactly q_floor.

TELEPORT IS A TRAINING SCAFFOLD, NOT PART OF THE READ (see teleport_at).
The teleport floor c > 0 is what keeps I - Q non-singular under gradient
pressure, but it displaces the g=0 bitwise-softmax corner by up to 2c in row
L1 -- a displacement that does NOT shrink with n. Measured at c=0.0125, A=[0],
logits ~ N(0,1) float64 under torch.manual_seed(0): max row L1 = 0.024870 at
n=16, 0.024965 at n=64, 0.024996 at n=256 -- rising toward the 2c ceiling
0.025, not falling.
The architecture's load-bearing claim -- that at g=0 the read IS the causal
softmax, bitwise -- therefore holds only at c = 0. Train with c > 0, anneal
c -> 0, EVALUATE AND SHIP at c = 0, where committor()'s singularity and
kappa_max guards are the live protection instead of the teleport. Self-check
(a) asserts both ends: bitwise at c=0, bounded displacement at c=0.0125.

Refusal, not a caught exception: state 0 has only j <= 0 available under
any causal mask, so P[0, 0] == 1 always. If index 0 is not declared
absorbing, Q has a 1 on its diagonal, I - Q is exactly singular, and the
chain is reducible (state 0 can never leave itself, and nothing reaches it).
committor() and q_floor_closed_form() both RAISE SingularTransientBlockError
in that case rather than returning a number. Non-finite input is the same
kind of refusal: a NaN logit RAISES ValueError instead of propagating a
full-NaN q with a NaN conditioning number that nothing downstream catches
((diag.abs() < tol).any() is False for NaN -- the hole this closes).
"""

import torch

__all__ = [
    "SingularTransientBlockError",
    "causal_mask",
    "build_operator",
    "teleport_at",
    "state_solve",
    "committor",
    "q_floor_closed_form",
    "TELEPORT",
    "KAPPA_DESIGN_BOUND_F32",
]


class SingularTransientBlockError(RuntimeError):
    """Raised when the transient block (I - Q) is singular, or so
    ill-conditioned that the solve would return garbage: a declared absorbing
    set fails to cover every self-absorbing / unreachable state, or an
    annealed teleport has let a row saturate."""


def causal_mask(n, device=None):
    """[n, n] bool, True where j <= i (causal, diagonal included)."""
    return torch.tril(torch.ones(n, n, dtype=torch.bool, device=device))


#: Teleport floor used DURING TRAINING. Every transient row sends at least this
#: much mass onto the causally-visible absorbing set, so ||Q||_inf <= 1 - TELEPORT
#: and hence ||(I-Q)^{-1}||_inf <= 1/TELEPORT = 80 BEFORE any training, making the
#: transient block non-singular by construction rather than by a threshold check:
#: diag(I-Q)_ii = 1 - Q_ii >= TELEPORT > 0 always.
#:
#: THE 80 IS NOT EXACT IN FLOAT32. The derivation assumes softmax rows sum to
#: exactly 1; in float32 they sum to 1 +/- 1.19e-07, so the max Q row sum reaches
#: 0.98750009 against the ideal 1 - c = 0.98750000 and the realized bound is
#:     ||(I-Q)^{-1}||_inf <= 80 * (1 + 6e-6) = 80.000480
#: Measured, exact norm, not the diagonal proxy: 80.000305 (absolute excess
#: 3.052e-04, relative 3.815e-06) worst over 2000 random float32 draws at logit
#: scale 1..100, n=16, nA=4; and 80.000214 over the eleven shipped stress
#: checkpoints x 512 Bed examples. The same construction in float64 reads
#: 79.995218912596 -- the excess is float32 rounding, nothing else.
#: A caller enforcing the design bound must test kappa <= 80 * (1 + 6e-6).
#: `assert kappa <= 80.0 + 1e-6` is measurably too tight: it FAILS on the
#: shipped lr_extreme checkpoint (80.000214) with the teleport fully intact.
TELEPORT = 0.0125

#: The float32-realized design bound at TELEPORT. Use this, not 80.0, in asserts.
KAPPA_DESIGN_BOUND_F32 = 80.0 * (1.0 + 6e-6)


def teleport_at(step, total_steps, start=TELEPORT, end=0.0):
    """Linear teleport anneal: `start` at step 0, `end` at step >= total_steps.

    The point of annealing rather than picking one end of the knob: containment
    and the bitwise corner are the SAME knob in opposite positions. c > 0 keeps
    I - Q non-singular while the logits are still moving; c = 0 is the only
    setting at which the g=0 read is the causal softmax bitwise.

    IS THE ANNEAL VIABLE? Measured on the eleven shipped stress checkpoints
    (ceqjepa/artifacts/stress_*.pt), 512 fresh Bed examples each, at teleport=0,
    counting transient rows with 1 - P[i,i] < nT*eps_float32:

        ten non-diverged runs (seeds 0-4, g=0.99, nA=1, nA=8, 10x lr, 100x lr):
            0 saturated rows out of 61,440. Worst slack 1 - P[i,i] = 4.053e-05
            (100x lr): the solve stays finite, but conditioning at c=0 is
            24,672 against 79.745 at c=0.0125 -- 309x worse.
        the diverged run (lr_extreme, max |logit| = 1.4e4):
            815 of 7,680 rows saturate (10.6%), 512 of 512 examples affected,
            min 1 - P[i,i] = 0.000e+00 EXACTLY.

    So: for a run whose logits stay at trained scale (max |logit| <= ~13) the
    anneal IS viable -- saturation frequency 0/61,440 -- but it is not free,
    because at c=0 nothing structural bounds kappa. Anneal only with
    committor(..., kappa_max=...) armed, and treat the raise as the run's
    verdict rather than an exception to swallow. For a diverged encoder the
    anneal is not viable at any schedule: every example saturates.
    """
    if total_steps <= 0:
        return float(end)
    f = min(max(float(step) / float(total_steps), 0.0), 1.0)
    return float(start) + (float(end) - float(start)) * f


def absorbing_teleport(n, absorbing_idx, dtype, device=None):
    """[n, n] row-stochastic teleport target: uniform over the absorbing states
    that are CAUSALLY VISIBLE from each row (a <= i).

    Rows with no visible absorbing state get a zero row and receive no teleport
    (the caller keeps the bare softmax there). With 0 in absorbing_idx every
    row i >= 1 has index 0 visible, which is why the canon declares the sink.
    """
    idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=device)
    A = torch.zeros(n, n, dtype=dtype, device=device)
    if idx.numel() == 0:
        return A
    rows = torch.arange(n, device=device).unsqueeze(1)          # [n,1]
    visible = (idx.unsqueeze(0) <= rows)                        # [n,k] a <= i
    A[rows.expand(-1, idx.numel()), idx.unsqueeze(0).expand(n, -1)] = visible.to(dtype)
    counts = A.sum(-1, keepdim=True)
    return torch.where(counts > 0, A / counts.clamp_min(1), A)


def build_operator(logits, absorbing_idx, teleport=TELEPORT):
    """Causal row-stochastic P from logits, with boundary rows overwritten.

    logits: [..., n, n]. absorbing_idx: 1-D long/int tensor or sequence of
    absorbing vertex indices. Returns P: [..., n, n].

    A `teleport` fraction of every transient row's mass is moved onto the
    causally-visible absorbing states BEFORE the boundary overwrite. This
    bounds ||Q||_inf <= 1 - teleport for every example at every training step,
    which is the structural repair for the singular-transient-block crash: no
    setting of the logits can drive P[i,i] to 1 once teleport > 0.

    teleport=0.0 is the SHIP/EVAL setting, not a curiosity: it is the only
    setting at which the g=0 read is the causal softmax bitwise (self-check
    (a)). At teleport=0 nothing structural protects the transient solve, so
    committor()'s guards are the protection -- keep them armed and give
    committor a kappa_max. See teleport_at() for the anneal and its measured
    cost.

    Raises ValueError on non-finite logits: a single NaN otherwise yields a
    full-NaN P, a full-NaN q and a NaN conditioning number that every
    downstream threshold check silently passes.
    """
    if not (0.0 <= teleport < 1.0):
        raise ValueError("teleport must be in [0, 1), got %r" % (teleport,))
    if not torch.isfinite(logits).all():
        n_bad = int((~torch.isfinite(logits)).sum())
        raise ValueError(
            "build_operator: logits contain %d non-finite entries (NaN/Inf); "
            "refusing to build P, because softmax would propagate NaN into q "
            "and into the conditioning number, where no threshold check "
            "catches it ((x < tol) is False for NaN)" % n_bad
        )
    n = logits.shape[-1]
    mask = causal_mask(n, device=logits.device)
    masked = logits.masked_fill(~mask, float("-inf"))
    P = torch.softmax(masked, dim=-1)
    absorbing_idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=logits.device)
    if teleport > 0.0 and absorbing_idx.numel() > 0:
        A = absorbing_teleport(n, absorbing_idx, P.dtype, P.device)
        has_target = (A.sum(-1, keepdim=True) > 0).to(P.dtype)   # [n,1]
        c = has_target * teleport                                # 0 where no visible target
        P = (1.0 - c) * P + c * A
    eye = torch.eye(n, dtype=P.dtype, device=P.device)
    P = P.clone()
    if absorbing_idx.numel() > 0:
        P[..., absorbing_idx, :] = eye[absorbing_idx]
    return P


def state_solve(P, V, g):
    """z = (I - g*P)^{-1} V by triangular forward substitution; O = (1-g)*P@z.

    P: [..., n, n] causal (lower triangular). V: [..., n, d]. g: scalar < 1.
    Returns (z, O), both [..., n, d].

    Needs no singularity guard at any teleport: diag(I - g*P)_ii = 1 - g*P_ii
    >= 1 - g > 0 for g < 1, whatever the logits do. Only the committor solve,
    where the coefficient is 1 rather than g, can go singular.
    """
    n = P.shape[-1]
    eye = torch.eye(n, dtype=P.dtype, device=P.device)
    M = eye - g * P
    z = torch.linalg.solve_triangular(M, V, upper=False)
    O = (1 - g) * (P @ z)
    return z, O


def committor(P, absorbing_idx, tol=None, kappa_max=None):
    """q^(bullet)_T = (I - Q)^{-1} R, embedded back to full [..., n, k].

    Raises SingularTransientBlockError if (I - Q)'s diagonal has a zero (a
    transient state whose only causal predecessor is itself), or if kappa_max
    is given and the EXACT conditioning ||(I-Q)^{-1}||_inf exceeds it. Raises
    ValueError if P is non-finite.

    kappa_max is the guard that matters once the teleport is annealed to 0:
    the diagonal test alone only catches saturation to within nT*eps, and a row
    at 1 - Q_ii = 4.053e-05 (measured, the 100x-lr checkpoint) passes it while
    returning a solve amplified 24,672x. Pass kappa_max=KAPPA_DESIGN_BOUND_F32
    to hold an annealed run to the conditioning the teleport used to guarantee.

    Journals committor.last_kappa_bound = ||(I-Q)^{-1}||_inf, EXACT rather than
    a bound: for lower-triangular Q >= 0 with Q_ii < 1, (I-Q)^{-1} = sum_k Q^k
    is entrywise nonnegative, so its infinity norm is exactly max_i (M^{-1} 1)_i
    -- one extra triangular solve against the ones vector. The value previously
    stashed there was 1 / min_i (1 - Q_ii), a LOWER bound that callers asserted
    against as if it were an upper bound. Measured understatement of that old
    quantity, exact / diagonal: 1.97x (uniform n=16 nA=4), 2.15x (uniform n=32
    A=[0,7,19]), 2.80x (uniform n=64 A=[0]: 1.9753 vs 5.5244), 3.45x (uniform
    n=256 A=[0]: 1.9753 vs 6.8092), and 1.00x..3.21x over 200 random scale-3
    operators at n=32. The float() runs detached, inside
    no_grad, so the per-step "Converting a tensor with requires_grad=True to a
    scalar" UserWarning is gone and the journal keeps no graph alive. With
    leading batch dimensions this is the max over the batch: one ill-conditioned
    example is visible but not attributable.
    """
    if not torch.isfinite(P).all():
        n_bad = int((~torch.isfinite(P)).sum())
        raise ValueError(
            "committor: P contains %d non-finite entries (NaN/Inf); refusing "
            "to solve, because the singularity test (diag.abs() < tol).any() "
            "is False for NaN, so a NaN P returns a full-NaN q and a NaN "
            "kappa with nothing raised" % n_bad
        )
    n = P.shape[-1]
    device = P.device
    absorbing_idx = torch.as_tensor(absorbing_idx, dtype=torch.long, device=device)
    is_absorbing = torch.zeros(n, dtype=torch.bool, device=device)
    is_absorbing[absorbing_idx] = True
    transient_idx = torch.nonzero(~is_absorbing, as_tuple=True)[0]  # ascending: preserves triangularity
    k = absorbing_idx.numel()

    Q = P[..., transient_idx, :][..., :, transient_idx]
    R = P[..., transient_idx, :][..., :, absorbing_idx]

    diag = 1.0 - torch.diagonal(Q, dim1=-2, dim2=-1)
    if tol is None:
        # Dtype-aware. A fixed absolute 1e-10 is BELOW float32 eps (1.192e-07),
        # so in float32 a saturated row passed the guard and committor() returned
        # a q whose channels summed to 1.1868 -- 1245x over the eps_32 = 1.5e-4
        # conservation bar, finite and inside [0,1] so nothing downstream caught
        # it. The guard must scale with the dtype and the transient dimension.
        tol = max(1e-10, transient_idx.numel() * torch.finfo(P.dtype).eps)
    if (diag.abs() < tol).any():
        raise SingularTransientBlockError(
            "transient block singular (I - Q has a diagonal entry below %.3e in %s): "
            "some causally-self-only state is not in the declared absorbing set, or "
            "an annealed teleport has let a row saturate. With "
            "build_operator(teleport=c>0) this is unreachable -- a teleport of c "
            "bounds every transient diagonal below by c." % (tol, P.dtype)
        )

    eyeT = torch.eye(transient_idx.numel(), dtype=P.dtype, device=device)
    M = eyeT - Q
    q_T = torch.linalg.solve_triangular(M, R, upper=False)

    # EXACT ||(I-Q)^{-1}||_inf by one more triangular solve, against 1.
    with torch.no_grad():
        Md = M.detach()
        ones = torch.ones(*Md.shape[:-1], 1, dtype=P.dtype, device=device)
        kappa = float(torch.linalg.solve_triangular(Md, ones, upper=False).abs().max())
    committor.last_kappa_bound = kappa
    if kappa_max is not None and kappa > kappa_max:
        raise SingularTransientBlockError(
            "transient block ill-conditioned: ||(I-Q)^{-1}||_inf = %.6g exceeds "
            "kappa_max = %.6g. At teleport=c the structural bound is 1/c (80 at "
            "c=0.0125, realized 80*(1+6e-6) in float32); at teleport=0 nothing "
            "bounds it and this guard is the only protection." % (kappa, kappa_max)
        )

    q_full = torch.zeros(*P.shape[:-1], k, dtype=P.dtype, device=device)
    q_full[..., transient_idx, :] = q_T
    q_full[..., absorbing_idx, :] = torch.eye(k, dtype=P.dtype, device=device)
    return q_full


def q_floor_closed_form(n, absorbing_idx, dtype=torch.float64, device=None,
                        teleport=TELEPORT):
    """Committor field of the uniform causal chain, closed form, no solve.

    With teleport c and A_vis(i) = {a in A : a <= i}, the uniform row is
        P[i,j] = (1-c)/(i+1)  for j <= i,   plus  c/|A_vis(i)|  for a in A_vis(i)
    so, writing S_{i-1} = sum_{j<i} q_j and u_i = mean of e_a over A_vis(i),
        q_i = [ (1-c)/(i+1) * S_{i-1} + c * u_i ] / (1 - (1-c)/(i+1))
    which is a forward recursion in causal order. At c = 0 this collapses to
    the bare q_i = mean(q_0 .. q_{i-1}).

    `teleport` must match the value build_operator was called with, or this is
    the floor of a different chain: under the anneal, pass teleport_at(step,...).

    Raises SingularTransientBlockError if index 0 is not absorbing (state 0
    is forced self-absorbing under any causal mask).
    """
    absorbing_list = sorted(int(a) for a in absorbing_idx)
    k = len(absorbing_list)
    pos = {a: j for j, a in enumerate(absorbing_list)}
    if 0 not in pos:
        raise SingularTransientBlockError(
            "index 0 is not in the declared absorbing set: state 0 is "
            "always self-absorbing under a causal mask (only j <= 0 exists)"
        )

    c = float(teleport)
    q = torch.zeros(n, k, dtype=dtype, device=device)
    running_sum = torch.zeros(k, dtype=dtype, device=device)
    for i in range(n):
        if i in pos:
            q[i, pos[i]] = 1.0
        else:
            w = (1.0 - c) / (i + 1)                    # uniform weight per visible j
            u = torch.zeros(k, dtype=dtype, device=device)
            vis = [j for a, j in pos.items() if a <= i]
            if vis and c > 0.0:
                u[vis] = 1.0 / len(vis)
            q[i] = (w * running_sum + c * u) / (1.0 - w)
        running_sum = running_sum + q[i]
    return q


## `ceqjepa.train` building blocks (inlined, `op.` calls point at the cell above)

In [ ]:
import os
import torch.nn as nn
import torch.nn.functional as F

EPS_Q = 1e-6  # BCE input clamp, MANDATORY per build spec: q attains 0 and 1 exactly
  # BCE input clamp, MANDATORY per build spec: q attains 0 and 1 exactly


# ---------------------------------------------------------------------------
# (a) synthetic in-class bed -- committor labels computed EXACTLY, via the
# shared operator's own exact solve (float64), never by the model.
# ---------------------------------------------------------------------------
class Bed:
    """A fixed causal absorbing-chain corpus over n positions, the first
    nA of which are absorbing (canonical order, matches build spec). Base
    tensors (L_env, D, W_obs, phi) are generated once at construction
    (seed 0) and fixed thereafter; each batch() draws fresh latents,
    builds the per-example causal operator via ceqjepa.operator, and
    solves the EXACT committor -- the label -- via ceqjepa.operator.committor
    in float64. Mirrors BUILD SPEC section 1 at a CPU-tiny geometry.
    """

    def __init__(self, n, nA, x_dim, z_dim, seed=0, dtype=torch.float64):
        assert n > nA >= 1
        self.n, self.nA = n, nA
        self.x_dim, self.z_dim = x_dim, z_dim
        self.dtype = dtype
        self.absorbing_idx = torch.arange(nA)
        g = torch.Generator().manual_seed(seed)
        self.L_env = torch.randn(n, n, generator=g, dtype=dtype)
        self.D = torch.randn(z_dim, n, n, generator=g, dtype=dtype) * 0.5
        self.W_obs = torch.randn(x_dim, z_dim, generator=g, dtype=dtype) / (z_dim ** 0.5)
        self.phi = torch.randn(n, x_dim, generator=g, dtype=dtype) / (x_dim ** 0.5)

    def batch(self, gen, B):
        """One batch, ONE solve (BUILD SPEC: 'Batch the solve'). Returns
        (x[B,x_dim] f32, x_nx[B,x_dim] f32, q_star[B,nA] f32, v_idx[B])."""
        n, nA = self.n, self.nA
        U = torch.randn(B, self.z_dim, generator=gen, dtype=self.dtype)
        v_idx = torch.randint(nA, n, (B,), generator=gen)
        logits = self.L_env.unsqueeze(0) + torch.einsum('bm,mij->bij', U, self.D)
        P = build_operator(logits, self.absorbing_idx)        # [B,n,n], causal, boundary rows overwritten
        q_full = committor(P, self.absorbing_idx)             # [B,n,nA], EXACT (I-Q)^-1 R
        rows = torch.arange(B)
        q_star = q_full[rows, v_idx]                             # [B,nA]  the label
        v_next = torch.multinomial(P[rows, v_idx], 1, generator=gen).squeeze(-1)
        noise = 0.05 * torch.randn(B, self.x_dim, generator=gen, dtype=self.dtype)
        x = torch.tanh(U @ self.W_obs.T + self.phi[v_idx]) + noise
        x_nx = torch.tanh(U @ self.W_obs.T + self.phi[v_next])
        return x.float(), x_nx.float(), q_star.float(), v_idx

    def q_floor(self):
        """(d) Closed-form committor of the uniform causal chain -- no
        encoder, no solve, no oracle. A collapsed encoder's read equals
        this exactly (WHAT SURVIVED, A)."""
        return q_floor_closed_form(self.n, self.absorbing_idx, dtype=torch.float32)


# ---------------------------------------------------------------------------
# The model: an encoder + chart-read wired on top of ceqjepa.operator's
# build_operator / committor / state_solve. The operator itself (causal
# softmax, boundary overwrite, resolvent solves) is never redefined here.
# ---------------------------------------------------------------------------
class TinyCEQ(nn.Module):
    def __init__(self, n, nA, d_enc, x_dim, z_dim_state, g, absorbing_idx, rank=8):
        super().__init__()
        self.n, self.nA, self.rank = n, nA, rank
        self.register_buffer('absorbing_idx', absorbing_idx)
        self.register_buffer('g', torch.tensor(float(g)))
        self.L0 = nn.Parameter(torch.zeros(n, n))               # spec: init ZEROS -> uniform causal chain at step 0
        self.Vt = nn.Parameter(torch.randn(n, z_dim_state) * (n ** -0.5))
        self.enc = nn.Sequential(nn.Linear(x_dim, d_enc), nn.GELU(), nn.Linear(d_enc, d_enc))
        # D6: per-example modulation of the shared chart, factored to the DCM-1 spec's own
        # rank r (default 8) instead of a dense [n,n] map. Two [d_enc -> n*r] linears produce
        # per-example factors a,b in R^{n,r}; delta_logits = a @ b.T is the rank-r perturbation.
        # Cost drops from O(d_enc*n*n) to O(d_enc*n*r).
        self.delta_a = nn.Linear(d_enc, n * rank)
        self.delta_b = nn.Linear(d_enc, n * rank)
        nn.init.zeros_(self.delta_a.weight)
        nn.init.zeros_(self.delta_a.bias)
        nn.init.zeros_(self.delta_b.weight)                     # spec: zero-init -> delta=0 at step 0, matches L0
        nn.init.zeros_(self.delta_b.bias)
        self.chart = nn.Linear(d_enc, n)                        # alpha read weights
        self.readout = nn.Linear(z_dim_state + nA, x_dim)

    def forward(self, x):
        B = x.shape[0]
        e = self.enc(x)                                                  # [B, d_enc]
        a = self.delta_a(e).view(B, self.n, self.rank)                   # [B,n,r]
        b = self.delta_b(e).view(B, self.n, self.rank)                   # [B,n,r]
        delta_logits = torch.einsum('bnr,bmr->bnm', a, b)                # [B,n,n], rank<=r
        logits = self.L0.unsqueeze(0) + delta_logits
        P = build_operator(logits, self.absorbing_idx)                # [B,n,n]
        q_field = committor(P, self.absorbing_idx)                    # [B,n,nA]
        Vt_b = self.Vt.unsqueeze(0).expand(B, -1, -1)
        z, O = state_solve(P, Vt_b, float(self.g))                    # [B,n,d]
        alpha = torch.softmax(self.chart(e), dim=-1)                     # [B,n]
        q_alpha = torch.einsum('bn,bnk->bk', alpha, q_field)             # THE SUPERVISED READ
        h = torch.einsum('bn,bnd->bd', alpha, O)
        x_hat = self.readout(torch.cat([h, q_alpha], dim=-1))
        return dict(q_alpha=q_alpha, q_field=q_field, h=h, x_hat=x_hat, alpha=alpha, P=P, e=e)


# ---------------------------------------------------------------------------
# (b) the loss, every term named (BUILD SPEC section 3, stage 1 only)
# ---------------------------------------------------------------------------
def stage1_loss(out, q_star, x_nx):
    qc = out['q_alpha'].clamp(EPS_Q, 1 - EPS_Q)
    L_q = -(q_star * qc.log() + (1 - q_star) * (1 - qc).log()).sum(-1).mean()
    L_z = F.mse_loss(out['x_hat'], x_nx)
    return L_q + 0.5 * L_z, L_q.item(), L_z.item()


# ---------------------------------------------------------------------------
# (c) + (d): eval -- constant-predictor control (C1) and collapse-floor diag
# ---------------------------------------------------------------------------
@torch.no_grad()
def evaluate(model, bed, q_bar, q_floor_table, gen, n, kappa_ceiling=80.0):
    x, x_nx, q_star, v_idx = bed.batch(gen, n)
    x, x_nx, q_star = x.to(q_bar.device), x_nx.to(q_bar.device), q_star.to(q_bar.device)
    out = model(x)
    # committor.last_kappa_bound is set by the committor() call inside model(x)
    # above (for q_field) -- the teleport guarantee is ||(I-Q)^-1||_inf <= 80(1+6e-6) in
    # float32 (D4). D8: a long soak can climb past kappa_ceiling while still healthy
    # (measured 75.6954 at step 1180, still climbing) -- a run must degrade LOUDLY, not
    # abort mid-flight, so this is a printed warning, never a raising assert.
    kappa_bound = committor.last_kappa_bound
    if kappa_bound > kappa_ceiling:
        print(f"[ceqjepa.train] WARNING: kappa_bound={kappa_bound:.6f} exceeded "
              f"--kappa-ceiling={kappa_ceiling} -- conditioning is degrading, continuing anyway")
    q_hat = out['q_alpha']
    mse_model = F.mse_loss(q_hat, q_star).item()
    mse_bar = F.mse_loss(q_bar.expand_as(q_star), q_star).item()   # (c) control
    S = 1.0 - mse_model / mse_bar if mse_bar > 0 else float('nan')  # the R-1 kill metric
    floor_here = q_floor_table[v_idx]                               # (d) same query positions the model saw
    collapse_floor = (q_hat - floor_here).abs().max().item()
    # var_across_examples = Var[e] taken over the BATCH dimension (dim=0) of the ENCODER
    # OUTPUT e = model.enc(x), averaged over the d_enc channels. This is the real collapse
    # detector, not collapse_floor/CFD above. CFD is monotone in logit scale only -- it
    # reads healthy even when the encoder ignores x entirely, as long as e still varies
    # across positions within one example. An encoder collapsed across the BATCH dimension
    # (identical e for every example, i.e. e independent of x) reads exactly 0 here
    # regardless of what CFD says.
    var_across_examples = out['e'].var(dim=0, unbiased=False).mean().item()
    loss, l_q, l_z = stage1_loss(out, q_star, x_nx)
    return dict(mse_model=mse_model, mse_bar=mse_bar, S=S,
                collapse_floor=collapse_floor, var_across_examples=var_across_examples,
                kappa_bound=kappa_bound, loss=loss.item(), L_q=l_q, L_z=l_z)


def save_checkpoint(path, model, args, n_params, q_bar, history, wall_s):
    """D7: atomic checkpoint write -- write to a temp path in the same dir, then
    os.replace() it onto `path`. os.replace is atomic on both POSIX and Windows,
    so a process killed mid-write leaves either the old checkpoint or nothing at
    `path`, never a half-written (corrupt) one."""
    tmp = path + '.tmp'
    torch.save(dict(
        model_state_dict=model.state_dict(),
        geometry=vars(args),
        n_params=n_params,
        q_bar=q_bar,
        final_eval=history[-1] if history else None,
        history=history,
        wall_s=wall_s,
    ), tmp)
    os.replace(tmp, path)


## Geometry -- DESIGN (matches `train.py --geometry design`: N=256, d_enc=128,
the frozen DCM-1 N and d; other dims stay at `train.py`'s own flag defaults).

`TinyCEQ`'s per-example `delta_logits` is the D6 low-rank factorization
(rank `RANK`, default 8: two `[d_enc -> n*rank]` linears, `delta_logits =
einsum('bnr,bmr->bnm', a, b)`, cost `O(d_enc*n*rank)`), matching `train.py`'s
own `TinyCEQ`. It still does not hit the ~325,792-param target exactly at
this N, d_enc, rank -- `n_params` below reports the actual count rather than
fudging it; a smaller `RANK` lands closer, left as a knob.

In [ ]:
N            = 256      # chart positions -- frozen DCM-1 design N
NA           = 4        # absorbing positions (train.py default, matches build spec)
D_ENC        = 128      # frozen DCM-1 design d
X_DIM        = 8        # train.py default
Z_DIM        = 4        # bed latent width, NOT known to the model (train.py default)
Z_DIM_STATE  = 6        # train.py default
G            = 0.9      # state-channel discount, frozen per build spec
RANK         = 3        # D6: delta_logits factorization rank (DCM-1 spec r=8)
BATCH_SIZE   = 64       # bumped from train.py's CPU default (16) since this runs on a T4
STEPS = 400
EVAL_EVERY = 100
EVAL_N = 512
LR           = 3e-4
SEED         = 0
KAPPA_CEILING = KAPPA_DESIGN_BOUND_F32  # D8: WARN above this, never assert/abort
CKPT_EVERY    = 500       # D7: atomic checkpoint every this many steps
CKPT_PATH     = "/kaggle/working/ceqjepa_dcm1_design.pt"  # Kaggle persists this dir as kernel output


In [ ]:
torch.manual_seed(SEED)

bed = Bed(N, NA, X_DIM, Z_DIM, seed=SEED)
train_gen = torch.Generator().manual_seed(SEED)
heldout_gen = torch.Generator().manual_seed(SEED + 1_000_000)  # disjoint stream, per build spec

# q_bar: the constant predictor's whole content -- training-set channel mean (control C1).
_, _, q_pool, _ = bed.batch(train_gen, max(2048, BATCH_SIZE))
q_bar = q_pool.mean(0).to(device)
q_floor_table = bed.q_floor().to(device)

model = TinyCEQ(n=N, nA=NA, d_enc=D_ENC, x_dim=X_DIM, z_dim_state=Z_DIM_STATE, g=G,
                rank=RANK, absorbing_idx=torch.arange(NA)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=LR, betas=(0.9, 0.999), eps=1e-8, weight_decay=0.01)

n_params = sum(p.numel() for p in model.parameters())
print(f"[ceqjepa] N={N} NA={NA} d_enc={D_ENC} x_dim={X_DIM} z_dim={Z_DIM} "
      f"z_dim_state={Z_DIM_STATE} g={G} rank={RANK} params={n_params} device={device}")
print(f"[ceqjepa] q_bar={[round(v, 4) for v in q_bar.tolist()]}")
print(f"[ceqjepa] target_params~325792 actual_params={n_params} "
      f"({'hit' if n_params == 325792 else 'did NOT hit'} the target exactly, reporting actual)")

# save_checkpoint() (inlined above from train.py) takes `args` and does vars(args) --
# a SimpleNamespace of the geometry config stands in for the argparse Namespace train.py has.
import types
_ckpt_args = types.SimpleNamespace(N=N, NA=NA, D_ENC=D_ENC, X_DIM=X_DIM, Z_DIM=Z_DIM,
                                    Z_DIM_STATE=Z_DIM_STATE, G=G, RANK=RANK,
                                    BATCH_SIZE=BATCH_SIZE, STEPS=STEPS, LR=LR, SEED=SEED)


## Training loop

Mirrors `train.py main()`'s loop, plus D7/D8: a periodic atomic checkpoint
to `CKPT_PATH` (`/kaggle/working`, so a 9-hour session timeout or any kill
leaves usable weights), and `kappa_bound` checked against `KAPPA_CEILING`
inside `evaluate()` as a printed WARNING, never an assert.
`SingularTransientBlockError` is not caught here: it means a genuine defect
(NaN logits, or an armed `kappa_max` at teleport=0) -- a bug to surface, not
a per-step event to paper over.

In [ ]:
history = []
t0 = time.time()

for step in range(1, STEPS + 1):
    x, x_nx, q_star, _ = bed.batch(train_gen, BATCH_SIZE)
    # Bed builds tensors on CPU (torch.randn with no device=); the model was moved
    # with .to(device). Without this the first forward raises a CPU/CUDA mismatch
    # at STEP 1 on a GPU -- the CPU smoke test cannot see it because both sides agree.
    x, x_nx, q_star = x.to(device), x_nx.to(device), q_star.to(device)
    x, x_nx, q_star = x.to(device), x_nx.to(device), q_star.to(device)
    model.train()
    out = model(x)
    loss, l_q, l_z = stage1_loss(out, q_star, x_nx)
    opt.zero_grad()
    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    if step % EVAL_EVERY == 0 or step == STEPS:
        model.eval()
        ev = evaluate(model, bed, q_bar, q_floor_table, heldout_gen, EVAL_N,
                       kappa_ceiling=KAPPA_CEILING)
        row = dict(step=step, train_loss=loss.item(), wall_s=time.time() - t0, **ev)
        history.append(row)
        print(f"[eval] step={step:6d} train_loss={loss.item():.4f} "
              f"S(vs const)={ev['S']:+.4f} mse_model={ev['mse_model']:.4e} "
              f"mse_bar={ev['mse_bar']:.4e} collapse_floor={ev['collapse_floor']:.4e} "
              f"var_across_examples={ev['var_across_examples']:.4e} "
              f"kappa_bound={ev['kappa_bound']:.4f}")

    # D7: periodic atomic checkpoint -- a killed run (9h Kaggle timeout, OOM, manual
    # stop) leaves the last one of these on disk under /kaggle/working instead of
    # nothing (the prior soak reached step 1840 healthy and left zero checkpoints).
    if CKPT_EVERY and step % CKPT_EVERY == 0:
        save_checkpoint(CKPT_PATH, model, _ckpt_args, n_params, q_bar, history,
                         time.time() - t0)
        print(f"[ceqjepa] wrote checkpoint {CKPT_PATH} at step={step}")

save_checkpoint(CKPT_PATH, model, _ckpt_args, n_params, q_bar, history, time.time() - t0)
print(f"[ceqjepa] wrote final checkpoint {CKPT_PATH}")
print(f"done in {time.time() - t0:.1f}s")


## Manifest

In [ ]:
manifest = dict(
    geometry=dict(N=N, NA=NA, d_enc=D_ENC, x_dim=X_DIM, z_dim=Z_DIM,
                  z_dim_state=Z_DIM_STATE, g=G, rank=RANK, batch_size=BATCH_SIZE,
                  steps=STEPS, lr=LR, seed=SEED),
    n_params=n_params,
    device=str(device),
    torch_version=torch.__version__,
    python_version=platform.python_version(),
    q_bar=q_bar.tolist(),
    wall_s=time.time() - t0,
    history=history,
    checkpoint_path=CKPT_PATH,
)
Path = __import__("pathlib").Path
Path("/kaggle/working/ceqjepa_kaggle_manifest.json").write_text(json.dumps(manifest, indent=2))
print("wrote /kaggle/working/ceqjepa_kaggle_manifest.json")
print("final S(vs const) =", history[-1]["S"] if history else None)
